# Clean CEO names

For every row of `data/ceos.csv`, look at `CEO` and `second_CEO`. If the name contains one or more middle initials (a capital letter followed by a period, e.g. `Stephen J. Hemsley`), build the version without the initials (`Stephen Hemsley`) and add it to the corresponding `CEO_alt_names` / `second_CEO_alt_names` column, unless it is already there.

Alternative names are stored as a `; `-separated list.

In [2]:
import re

import pandas as pd

CSV_PATH = "data/ceos.csv"

# keep everything as text so empty cells stay empty strings instead of NaN
df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
print(f"Loaded {len(df)} rows from {CSV_PATH}")
df.head()

Loaded 3000 rows from data/ceos.csv


,year,company,company_alt_names,rank,CEO,CEO_alt_names,second_CEO,second_CEO_alt_names,wiki_page_snapshot,wiki_page_live
0,2025,Walmart,,1,Doug McMillon,Douglas McMillon,,,https://en.wikipedia.org/w/index.php?title=Wal...,https://en.wikipedia.org/wiki/Walmart
1,2025,Amazon,Amazon.com,2,Andy Jassy,Andrew Jassy; Andrew R. Jassy,,,https://en.wikipedia.org/w/index.php?title=Ama...,https://en.wikipedia.org/wiki/Amazon_(company)
2,2025,UnitedHealth Group,,3,Stephen J. Hemsley,Stephen Hemsley,,,https://en.wikipedia.org/w/index.php?title=Uni...,https://en.wikipedia.org/wiki/UnitedHealth_Group
3,2025,Apple,,4,Tim Cook,Timothy D. Cook,,,https://en.wikipedia.org/w/index.php?title=App...,https://en.wikipedia.org/wiki/Apple_Inc.
4,2025,CVS Health,,5,David Joyner,,,,https://en.wikipedia.org/w/index.php?title=CVS...,https://en.wikipedia.org/wiki/CVS_Health


## Helpers

In [3]:
# a single capital letter followed by a period, not preceded by another letter
# (so "J." in "Stephen J. Hemsley" matches, and both "D." and "G." in "D.G. Macpherson" do too)
INITIAL_RE = re.compile(r"(?<![A-Za-z])[A-Z]\.")


def has_initial(name):
    """True if the name contains at least one initial followed by a period."""
    return bool(INITIAL_RE.search(name))


def strip_initials(name):
    """Remove every initial+period from a name and tidy up the leftover spaces."""
    return re.sub(r"\s+", " ", INITIAL_RE.sub("", name)).strip()


def split_alt_names(cell):
    """Turn a `; `-separated cell into a list of names."""
    return [part.strip() for part in cell.split(";") if part.strip()]


def join_alt_names(names):
    """Turn a list of names back into a `; `-separated cell."""
    return "; ".join(names)


# quick sanity check
for example in ["Stephen J. Hemsley", "W. Randall Fowler", "Sean M. O'Connor", "Doug McMillon"]:
    print(f"{example!r:25} -> has_initial={has_initial(example)!s:5} stripped={strip_initials(example)!r}")

'Stephen J. Hemsley'      -> has_initial=True  stripped='Stephen Hemsley'
'W. Randall Fowler'       -> has_initial=True  stripped='Randall Fowler'
"Sean M. O'Connor"        -> has_initial=True  stripped="Sean O'Connor"
'Doug McMillon'           -> has_initial=False stripped='Doug McMillon'


## Clean the columns

A stripped name is only added when it still has at least two words. Names such as `J. Alexander` or the truncated `Thomas E.` would collapse to a single word, so they are reported and left alone.

In [4]:
COLUMN_PAIRS = [("CEO", "CEO_alt_names"), ("second_CEO", "second_CEO_alt_names")]

added_count = 0
already_present_count = 0
skipped = []

for index, row in df.iterrows():
    for name_col, alt_col in COLUMN_PAIRS:
        name = row[name_col].strip()
        if not name or not has_initial(name):
            continue

        stripped = strip_initials(name)

        # a name that loses its first name ("J. Alexander" -> "Alexander") is not usable
        if len(stripped.split()) < 2:
            skipped.append((index, name_col, name, stripped))
            print(f"[skip] row {index} {name_col}: {name!r} -> {stripped!r} (fewer than two words)")
            continue

        alt_names = split_alt_names(row[alt_col])
        existing = {alt.casefold() for alt in alt_names} | {name.casefold()}

        if stripped.casefold() in existing:
            already_present_count += 1
            continue

        alt_names.append(stripped)
        df.at[index, alt_col] = join_alt_names(alt_names)
        added_count += 1
        print(
            f"[add ] row {index} {name_col}: {name!r} -> added {stripped!r} to {alt_col} "
            f"(now: {df.at[index, alt_col]!r})"
        )

print()
print(f"Total CEO names added: {added_count}")
print(f"Already present, nothing to do: {already_present_count}")
print(f"Skipped (stripped name too short): {len(skipped)}")


Total CEO names added: 0
Already present, nothing to do: 1224
Skipped (stripped name too short): 0


## Remove the `Jr.` / `Sr.` suffixes

Same idea as above, but this time for the generational suffixes. The name is stripped of **both** the suffix and any initials, so `H. Lawrence Culp Jr.` yields `Lawrence Culp`.

The match is case insensitive and tolerates the variants that show up in this kind of data: `Jr.`, `Jr`, `JR.`, and a comma before the suffix (`John Turner, Jr.`).

In [5]:
# an optional comma, then "jr"/"sr" as a whole word, with an optional period
SUFFIX_RE = re.compile(r",?\s*\b(?:jr|sr)\b\.?", re.IGNORECASE)


def has_suffix(name):
    """True if the name ends with a Jr./Sr. style generational suffix."""
    return bool(SUFFIX_RE.search(name))


def strip_suffix_and_initials(name):
    """Remove Jr./Sr. suffixes *and* any initials, then tidy up the spaces."""
    without_suffix = SUFFIX_RE.sub("", name)
    return re.sub(r"\s+", " ", INITIAL_RE.sub("", without_suffix)).strip()


# quick sanity check
for example in ["H. Lawrence Culp Jr.", "John Turner, Jr.", "Sam Walton SR", "Doug McMillon"]:
    print(f"{example!r:25} -> has_suffix={has_suffix(example)!s:5} cleaned={strip_suffix_and_initials(example)!r}")

suffix_added_count = 0
suffix_already_present_count = 0
suffix_skipped = []

print()
for index, row in df.iterrows():
    for name_col, alt_col in COLUMN_PAIRS:
        name = row[name_col].strip()
        if not name or not has_suffix(name):
            continue

        cleaned = strip_suffix_and_initials(name)

        # a name reduced to a single word is not usable
        if len(cleaned.split()) < 2:
            suffix_skipped.append((index, name_col, name, cleaned))
            print(f"[skip] row {index} {name_col}: {name!r} -> {cleaned!r} (fewer than two words)")
            continue

        alt_names = split_alt_names(row[alt_col])
        existing = {alt.casefold() for alt in alt_names} | {name.casefold()}

        if cleaned.casefold() in existing:
            suffix_already_present_count += 1
            continue

        alt_names.append(cleaned)
        df.at[index, alt_col] = join_alt_names(alt_names)
        suffix_added_count += 1
        print(
            f"[add ] row {index} {name_col}: {name!r} -> added {cleaned!r} to {alt_col} "
            f"(now: {df.at[index, alt_col]!r})"
        )

print()
print(f"Total CEO names added (Jr./Sr. removed): {suffix_added_count}")
print(f"Already present, nothing to do: {suffix_already_present_count}")
print(f"Skipped (cleaned name too short): {len(suffix_skipped)}")

'H. Lawrence Culp Jr.'    -> has_suffix=True  cleaned='Lawrence Culp'
'John Turner, Jr.'        -> has_suffix=True  cleaned='John Turner'
'Sam Walton SR'           -> has_suffix=True  cleaned='Sam Walton'
'Doug McMillon'           -> has_suffix=False cleaned='Doug McMillon'

[add ] row 117 CEO: 'H. Lawrence Culp Jr.' -> added 'Lawrence Culp' to CEO_alt_names (now: 'Lawrence Culp Jr.; Lawrence Culp')
[add ] row 138 CEO: 'Michael C. Creedon Jr.' -> added 'Michael Creedon' to CEO_alt_names (now: 'Michael Creedon Jr.; Michael Creedon')
[add ] row 167 CEO: 'William H. Rogers Jr.' -> added 'William Rogers' to CEO_alt_names (now: 'William Rogers Jr.; William Rogers')
[add ] row 196 CEO: 'James C. Fish Jr.' -> added 'James Fish' to CEO_alt_names (now: 'James Fish Jr.; James Fish')
[add ] row 292 CEO: 'Frank B. Holding Jr.' -> added 'Frank Holding' to CEO_alt_names (now: 'Frank Holding Jr.; Frank Holding')
[add ] row 366 CEO: 'J. Patrick Gallagher Jr.' -> added 'Patrick Gallagher' to CEO_alt_na

## Save

In [6]:
df.to_csv(CSV_PATH, index=False)
print(f"Saved {len(df)} rows to {CSV_PATH}")

Saved 3000 rows to data/ceos.csv
